# 3.2 Building Agents with Aviary

In Chapter 3.1, we described an agent loop: a language model chooses an action, software runs it, and the results becomes an observation that informs the next choice. In this notebook, we will build a small version of that loop for a protein research question using [Aviary](https://github.com/Future-House/aviary), an extensible gymnasium for defining agent environments and [LDP](https://github.com/Future-House/ldp) a framework for defining language agents. With these packages you have more freedom to customize agents and use open source language models.

[Aviary](https://github.com/Future-House/aviary) is an **agent gym:** a framework for creating tasks that an agent can interact with. In this context, "gym" refers to a controlled setting where we define the goal, the actions available, what the agent can observe, and when a run ends. An Aviary environment holds the task's internal state and makes tools available. Its `reset()` method starts a run and provides the first observations and tools. Each time the agent requests a tool, `step()` executes the action and returns new observations, a reward value, and flags indicating whether the run has ended or been cut short. The agent may see only the observations, not the environment's entire internal state.

[LDP](https://github.com/Future-House/ldp), short for *Language Decision Process*, provides the other side of the interaction. We use it to define an **agent** that passes observations and available tools to a language model, receives its choosen action, and keeps track of the messages needed for the next decision. LDP's `RolloutManager` runs the repeated agent-environment interaction; one complete attempt is called a **rollout**. In this notebook, the model makes the choices, while the surrounding code runs the loop.

The goal of this tutorial is to build a small agent agent that can choose tools to calculate properties of a supplied protein sequence, summarize a protein's biological role, and propose questions for further drug-discovery research. We will give the environment tools for calculating properties of a protein sequence and summarizing a protein's biological role, then follow the actions the agent takes.  We will follow its tool calls to see how the model's decisions, the environment, and the code running the loop fit together. Once you understand this basic workflow, you will be able to design and build your own agents for other scientific tasks. All FutureHouse agents, including [PaperQA2](https://github.com/Future-House/paper-qa) and [Finch](https://github.com/Future-House/finch), are implemented using Aviary and LDP. Aviary and LDP also support training and evaluation workflows, but this example simply runs an agent; it does not train the model. A reward returned by an environment is a value we definte for that task, not automatically a measure of scientific correctness.

<figure style="text-align: center;">
  <img src="https://raw.githubusercontent.com/Future-House/tutorial-series/main/figures/Aviary.png" 
       alt="Agent action and observation loop" 
       style="max-width: 100%;">
  <figcaption><em>Figure 3.2.1: An agent iteratively received observations from the environment, takes an action based on the observation until the task is completed.</em></figcaption>
</figure>


<details> 
<summary> 🚀 How to run the notebook </summary>

This tutorial can be launched using the rocket (🚀) button at the top of the page.

**Option 1 — Google Colab (recommended)**
Opens the notebook in Google Colab with the fastest and most reliable experience.

Before running the tutorial, add your API keys using **either**:

- a `.env` file, or
- **Colab Secrets** (`🔑 Secrets` tab in the left sidebar)

Example `.env`:
```bash
OPENAI_API_KEY=your_key_here
ANTHROPIC_API_KEY=your_key_here
```

**Option 2 — MyBinder**
Launches a temporary cloud Jupyter environment directly in your browser.

⚠️ Binder environments can take a few minutes to build and start.

After the notebook loads, create a `.env` file in the notebook directory containing your API keys:

```bash
OPENAI_API_KEY=your_key_here
ANTHROPIC_API_KEY=your_key_here
```

**Notes**
- You only need API keys for the providers used in a given notebook.
- Never commit or publicly share your API keys.
- If a cell fails due to missing credentials, verify that your keys were loaded correctly before rerunning the cell.
</details>


### Before you begin

This example uses an OpenAI model to choose actions and summarize a protein's role. Install the packages below if you are running the notebook in a fresh environment. If you launched the tutorial through Binder, its requirements are already installed.


In [1]:
%pip install --upgrade fhaviary ldp pydantic openai biopython==1.86

  Using cached openai-3.19.2-py3-none-any.whl.metadata (44 kB)
  Using cached httpx2-2.13.1-py3-none-any.whl.metadata (9.8 kB)
  Using cached httpcore2-2.13.1-py3-none-any.whl.metadata (26 kB)
  Using cached truststore-0.10.4-py3-none-any.whl.metadata (4.4 kB)

[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


The model calls require an OpenAI API key. In Colab, add a secret named `OPENAI_API_KEY` and grant this notebook access to it. In another Jupyter environment, set the `OPENAI_API_KEY` environment variable or place it in a local `.env` file.

Run the next cell to load the key. The OpenAI SDK can read the key from that environment variable.

In [2]:
import os

LLM_API_KEYS = {
    "openai":    "OPENAI_API_KEY",
    "anthropic": "ANTHROPIC_API_KEY",
}

def get_api_key(llm: str = "openai") -> str:
    """
    Load API key for the specified LLM from Colab secrets,
    environment variable, or user input.
    
    Args:
        llm: LLM provider name. eg: 'openai', 'anthropic'
    
    Returns:
        API key string
    
    Example:
        api_key = get_api_key("anthropic")
    """

    llm = llm.lower()
    if llm not in LLM_API_KEYS:
        raise ValueError(
            f"Unknown LLM '{llm}'. Choose from: {list(LLM_API_KEYS.keys())}"
        )

    env_var = LLM_API_KEYS[llm]

    # 1. Try Colab secrets
    try:
        from google.colab import userdata
        key = userdata.get(env_var)
        if key:
            return key
    except ImportError:
        pass

    # 2. Try environment variable / .env file
    try:
        from dotenv import load_dotenv
        load_dotenv()
        key = os.environ.get(env_var)
        if key:
            return key
    except ImportError:
        pass

    raise ValueError(
        f"API key not found. Please set {env_var}:\n"
        f"  export {env_var}='your-key-here'\n"
        f"  or add it to a .env file"
    )

# Set the API key as an environment variable
os.environ["OPENAI_API_KEY"] = get_api_key("openai")
print("OpenAI API key loaded!")

OpenAI API key loaded!


## 3.2.2 Defining the Agent's Tools

As discussed previously, a tool is a function the agent can choose to call. In this example, we'll build **four tools**:
- `get_protein_sequence` – retreives the amino acid sequence for a protein
- `analyze_protein_sequence` — calculates properties from an amino acid sequence
- `summarize_protein_role` — asks a language model for a short summary of a protein's biological role
- `submit_final_answer` - submits the final answer to the user

Aviary turns these Python functions into tools using their **tool schema:** their names, input types, and dosctrings. Those descriptions help the agent decide which tool to request and what arguments to provide. The agent chooses a call, then the environment executes the function and returns its output as an observation.

An important note to add is that when you define a tool with `Aviary` it must contain a docstring with function description. See the example tool definitions below.

> Here are a few brief definitions of key classes and concepts from [Aviary](https://github.com/Future-House/aviary) and  [LDP](https://github.com/Future-House/ldp) for reference:
>
> **From Aviary**
>
> - **Message:** Used by language agents and environments for communication. Messages include attributes like content ot role (system, user, assistant, tool ), matching OpenAI's conventions.
> - **Environment:** An environment is a stateful system or "world" where an agent operates by taking actions. In Aviary, these actions are called tools. The environment presents states that the agent observes (totally or partially), prompting it to use tools to affect outcomes. Each action taken yields a reward and leads to a new state.
> - **Tool:** Defines an environmental tool that an agent can use to accomplish its task. Each environment contains its own set of tools. Most tools take arguments and tools can be called in parallel.
> - **ToolRequestMessage:** This is a specialized subclasses of Message used for tool requests. Typically, a language agent sends a ToolRequestMessage to the environment to request the execution of a specific tool. The role of ToolRequestMessage is always assistant.
>
> **From LDP**
>
> - **Agent:** An entity that interacts with the environment, mapping observations to tool request actions.
> - **Op:** Represents an operation within the agent. LDP includes various operations (Ops), such as API LLM calls, API embedding calls, or PyTorch module handling. These operations form the compute graph.
> - **OpResult:** the output of an Op.

Now let's write the python functions to define the two tools.

In [26]:
import json

from Bio.SeqUtils.ProtParam import ProteinAnalysis
from openai import OpenAI
from urllib.request import urlopen

# ── TOOL 1: Retrieve the amino acid sequence for a protein ──────────────────────────────────────

def get_protein_sequence(pdb_id: str, entity_id: int) -> dict:
    """Retrieve the amino acid sequence of one protein entity from RCSB PDB.

    Args:
        pdb_id: PDB structure identifier, such as 9RIP.
        entity_id: Protein entity number within the structure.
    """
    pdb_id = pdb_id.upper()
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}"

    with urlopen(url, timeout=15) as response:
        record = json.load(response)

    polymer = record["entity_poly"]
    if polymer["rcsb_entity_polymer_type"] != "Protein":
        raise ValueError(f"Entity {entity_id} is not a protein.")

    return {
        "pdb_id": pdb_id,
        "entity_id": entity_id,
        "protein_name": record["rcsb_polymer_entity"]["pdbx_description"],
        "sequence": polymer["pdbx_seq_one_letter_code_can"],
        "source": url,
    }

# ── TOOL 2: Analyze a protein sequence ──────────────────────────────────────

def analyze_protein_sequence(sequence: str) -> dict:
    """
    A tool to analyze a protein sequence.
    Use when you need to get basic biophysical properties of a protein,
    e.g., molecular weight, isoelectric point, instability index, GRAVY score, etc.

    Args:
        sequence: The protein sequence to analyze
    Returns:
        A dictionary containing the biophysical properties of the protein.
    """

    sequence = sequence.upper().strip()
    analysis = ProteinAnalysis(sequence)

    results = {
        "length": len(sequence),
        "molecular_weight_Da": round(analysis.molecular_weight(), 2),
        "isoelectric_point": round(analysis.isoelectric_point(), 2),
        "instability_index": round(analysis.instability_index(), 2),
        "gravy_score": round(analysis.gravy(), 3),   # Hydrophobicity
        "amino_acid_percent":  {
            aa: round(pct, 1)
            for aa, pct in analysis.amino_acids_percent.items()
            if pct > 0   # Only show amino acids actually present
        },
    }

    # Interpret some values for the non-expert
    results["is_stable"] = results["instability_index"] < 40
    results["is_hydrophilic"] = results["gravy_score"] < 0

    return results

# ── TOOL 3: Summarize protein biological role ────────────────────────────────

def summarize_protein_role(protein_name: str, organism: str, protein_data: dict | None = None) -> str:
    """
    A tool to summarize the biological
    role of a protein from its training knowledge.
    eg: biological function, disease or condition it is associated with, why it is considered a drug target.

    Args:
        protein_name: The name of the protein to summarize
        organism: The organism the protein belongs to
        protein_data: A dictionary containing the protein data
    Returns:
        A string containing the summary of the protein's biological role.
    """

    client = OpenAI(api_key=get_api_key(llm="openai"))
    response = client.responses.create(
    model="gpt-4.1-nano-2025-04-14",
    input= (
                f"Provide a concise 3–4 sentence summary of the protein '{protein_name}' "
                f"in {organism}. Here is the protein data: {protein_data}\n. Cover: (1) its biological function, "
                f"(2) which disease or condition it is associated with, "
                f"(3) why it is considered a drug target. Be factual and concise."
            ),
    )
    
    return response.output_text

# ── TOOL 4: Submit the final answer ────────────────────────────────

def submit_final_answer(answer: str) -> str:  # noqa: RUF029
    """
    A tool to submit the final answer to the user.

    Args:
        answer: The answer to the query.
    Returns:
        True if the answer is submitted, False otherwise
    """

    return answer

## 3.2.2 Defining the Environment

The tools are Python functions, but the agent needs an **environment** to make them available and execute its requests. We will define an Aviary environment for one protein research question.

The environment stores the question and whether the agent has submitted an answer. Its `reset()` method starts an attemp and returns the intial messages and available tools. When the agent requests a tool, `step()` executes the request and returns the tool's response as a new observation. It also reports a reward, whether the attempt is complete (`done`), and whether it was stopped early (`truncated`). 

The environment's **state** is information kept by the program across steps. The agent does not automatically see every state field; it acts on the messages and tools it receives.

Next, we will define a simple `state` and `environment` where an agent takes actions to modify analyze a protein.

>💡 *Reminders*
>
>**Environment** = the task setup that provides tools, executes the agent's requested actions, and returns observations
>
>**State** = information the environment keeps between actions, such as the question, submitted answer, and completion status
>
>**Observation** = information the environment sends to the agent, such as intial question or tool result


In [28]:
from typing import cast
from aviary.core import (
    Environment,
    Message,
    Messages,
    Tool,
    ToolRequestMessage,
    ToolResponseMessage,
)
from pydantic import BaseModel


# Defines the instructions given to the LLM at the start of a conversation
SYSTEM_PROMPT = """
You are an expert researcher. You are given a research question, Your task is to answer the question. You have access to the following tools:
- get_protein_sequence: to get the protein sequence
- analyze_protein_sequence: to analyze the protein sequence
- summarize_protein_role: to summarize the biological role of the protein
- submit_final_answer: to submit the final answer
"""

class DemoEnvState(BaseModel):
    """State of the EvalAgent."""

    query: str
    answer: str | None = None
    done: bool = False

class DemoAgentEnv(Environment[DemoEnvState]):
    """Environment for the DemoAgent."""

    def __init__(
        self,
        query: str, # The input to the agent
    ):
        self.query = query
        self.tools: list[Tool] = []
        self.messages: Messages | None = None

    def make_initial_state(self) -> DemoEnvState:
        """
        This initializes the state of the agent,
        i.e., where the agent at the beginning of the task.
        You can add more fields to the state if you want.
        """
        return DemoEnvState(
            query=self.query,
            answer=None,
            done=False
        )
    
    async def reset(self) -> tuple[Messages, list[Tool]]:
        """
        Reset the environment and collect initial observation(s).
        Possible observations could be instructions on how tools are related,
        or the goal of the environment.
        Should return a two-tuple of initial observations and tools.
        """
        self.messages = [
            Message(content=SYSTEM_PROMPT, role="system"),
            Message(content=self.query, role="user"), # User query goes here, not in system prompt
        ]
        self.tools = [
            Tool.from_function(get_protein_sequence),
            Tool.from_function(analyze_protein_sequence),
            Tool.from_function(summarize_protein_role),
            Tool.from_function(submit_final_answer),
        ] # Converts the Python functions into Aviary Tool objects using Tool.from_function()

        self.state = self.make_initial_state()
        return self.messages, self.tools
    
    async def step(self, action: ToolRequestMessage) -> tuple[Messages, float, bool, bool]:
        """
        Accepts an action from the agent and executes it.
        Returns: observations, reward, done, truncated.
        """
        response_messages = cast(
            "Messages",
            await self.exec_tool_calls(
                action,
                concurrency=False,
                handle_tool_exc=True,
                state=self.state,
            ),
        ) or [Message(content=f"No tool calls input in tool request {action}.")]

        done = any(
            isinstance(msg, ToolResponseMessage)
            and msg.name == submit_final_answer.__name__
            for msg in response_messages
        )
        self.intermediate_answer = response_messages[-1].content
        if done:
            self.state.done = True

        return (
            response_messages,
            1 if self.state.done else 0,
            self.state.done,
            False,
        )

## 3.2.3 Initializing the LDP Agent

The environment can now execute tools, but it does not choose which tool to use. We will define an LDP agent that sends the current observations and available tools to a language model, recevies the model's chosen action, and keeps the conversation history needed for its next decision.

Here, `AgentState` belongs to the agent. It stores the tools and previous messages. This is separate from `DemoEnvState`, which belongs to the environment and stores the task's question, answer, and completion status.

The `ldp` package has pre-defined Simple and ReAct agents which you can implement. You can refer to this [GitHub repo](https://github.com/Future-House/ldp) for more details on agent implementation.

In [29]:
from pydantic import BaseModel, Field
from ldp.agent import Agent
from ldp.graph import LLMCallOp

from aviary.core import ToolRequestMessage
from aviary.core import Message, Tool

from lmi.config import LLMConfig, ModelSpec


class AgentState(BaseModel):
    """Simple bucket to store available tools and previous messages."""

    tools: list[Tool] = Field(default_factory=list)
    messages: list[Message] = Field(default_factory=list)


class SimpleAgent(Agent):
    def __init__(self, **kwargs: dict) -> None:
        """Accepts config args and passes them to LLMCallOp."""
        self._llm_call_op = LLMCallOp(**kwargs)

    async def init_state(self, tools: list[Tool]) -> AgentState:
        """Receives tools and stores them."""
        return AgentState(tools=tools)

    async def get_asv(self, agent_state: AgentState, obs: list[Message]) -> tuple[ToolRequestMessage, AgentState, float]:
        """Take an action, observe new state, return value."""
        action: ToolRequestMessage = await self._llm_call_op(
            config=LLMConfig(
                models=[ModelSpec.from_name("gpt-4o-mini", temperature=0.1)]
            ), # LLMCallOp expects an LLMConfig object
            msgs=agent_state.messages + obs,
            tools=agent_state.tools,
        )
        new_state: AgentState = AgentState(
            messages=agent_state.messages + obs + [action.value],
            tools=agent_state.tools,
        )
        # Return action, state, value
        return action, new_state, 0.0

## 3.2.4 Running the Agent

We can now put the pieces together. LDP's `RolloutManager` repeatedly asks the agent to choose an action and passes the action to the Aviary environment. The resulting trajectory records what the agent requested and what the environment returned.

We will run one environment with a limit of five actions (`max_steps = 5`). The run ends sooner if the agent submits a final answer, If it reaches the limit first, then the trajectory is marked as truncated.

In [30]:
from ldp.alg import RolloutManager

# Initate the agent
agent = SimpleAgent()
runner = RolloutManager(agent=agent)

# Define the query
query = (
    "What is the biological role of the protein with PDB id 9RIP? "
    "Please analyze this protein and help me think about potential small-molecule drug targeting strategies."
)

# Perform rollouts
trajectories: list[tuple] = await runner.sample_trajectories(
    environments=[DemoAgentEnv(query=query)], # Must be a list of environments
    max_steps = 5, # Max number of steps to run for each environment
)

The rollout produced one trajectory because we provided one environment. Each step records the agent's requested tool calls and the observations returned by the environment. You can inspect them to see whether the agent retrieved a specific protein entity's sequence before passing it to `analyze_protein_sequence`.

Next, we'll print the answer passed to `submit_final_answer`. We use `steps[-1]` because a successfully completed run ends with that tool call. We use `trajectories[-1]` to select the last trajectory. We could also use `trajectory[0]` because this run only has one trajectory, so `trajectories[-1]` and `trajectory[0]` refer to the same run.

To run two questions, pass two environments to `sample_trajectories()`. The result will contain a trajectory for each environment.

```
trajectories: list[tuple] = await runner.sample_trajectories(
    environments=[
    DemoAgentEnv(query=query_1),
    DemoAgentEnv(query=query_2),
    ],
    max_steps = 5
)
```

In [31]:
# Print the tool calls from the last step of the last trajectory
tool_calls = trajectories[-1].steps[-1].action.value.tool_calls

# Get the answer from submit_final_answer tool call
for tc in tool_calls:
    if tc.function.name == "submit_final_answer":
        print(tc.function.arguments["answer"])

The protein with PDB id 9RIP is the HIV-1 Genome polyprotein. It has a length of 297 amino acids and a molecular weight of approximately 32.7 kDa. The isoelectric point is 6.13, and it has an instability index of 44.73, indicating that it is not very stable. The GRAVY score is -0.356, suggesting that it is hydrophilic in nature.

Biologically, the HIV-1 Genome polyprotein is a large precursor that is cleaved to produce essential viral proteins required for viral replication and assembly. Its processing is critical for the maturation of infectious virions, making it a key player in the HIV life cycle. 

Given its central role in the viral life cycle, the HIV-1 Genome polyprotein is considered a prime drug target. Small-molecule drug targeting strategies could include the development of protease inhibitors that block the cleavage of this polyprotein, thereby inhibiting the maturation of the virus and its ability to replicate. This approach has been successfully utilized in existing antir

### Inspect the Agent's Steps

The final answer shows what the agent submitted. The trajectory shows how it got there. Check that `get_protein_sequence` returned a sequence for the intended protein entity and that `analyze_protein_sequence` received that sequence. The environmment's `next_observation` contains each tool's response.

In [33]:
for i, step in enumerate(trajectories[-1].steps, start=1):
    print(f"\nStep {i} — requested tools")
    for call in step.action.value.tool_calls:
        print(f"  {call.function.name}: {call.function.arguments}")

    print("Tool responses:")
    for message in step.next_observation:
        print(f"  {getattr(message, 'name', type(message).__name__)}:")
        print(f"  {message.content}")


Step 1 — requested tools
  get_protein_sequence: {'pdb_id': '9RIP', 'entity_id': 1}
Tool responses:
  get_protein_sequence:
  {"pdb_id": "9RIP", "entity_id": 1, "protein_name": "Genome polyprotein", "sequence": "GDRVADVIESSIGDSVSRALTQALPAPTGQNTQVSSHRLDTGEVPALQAAEIGASSNTSDESMIETRCVLNSHSTAETTLDSFFSRAGLVGEIDLPLEGTTNPNGYANWDIDITGYAQMRRKVELFTYMRFDAEFTFVACTPTGQVVPQLLQYMFVPPGAPKPDSRESLAWQTATNPSVFVKLTDPPAQVSVPFMSPASAYQWFYDGYPTFGEHKQEKDLEYGACPNNMMGTFSVRTVGSSKSKYPLVVRIYMRMKHVRAWIPRPMRNQNYLFKANPNYAGNSIKPTGTSRTAITTL", "source": "https://data.rcsb.org/rest/v1/core/polymer_entity/9RIP/1"}

Step 2 — requested tools
  analyze_protein_sequence: {'sequence': 'GDRVADVIESSIGDSVSRALTQALPAPTGQNTQVSSHRLDTGEVPALQAAEIGASSNTSDESMIETRCVLNSHSTAETTLDSFFSRAGLVGEIDLPLEGTTNPNGYANWDIDITGYAQMRRKVELFTYMRFDAEFTFVACTPTGQVVPQLLQYMFVPPGAPKPDSRESLAWQTATNPSVFVKLTDPPAQVSVPFMSPASAYQWFYDGYPTFGEHKQEKDLEYGACPNNMMGTFSVRTVGSSKSKYPLVVRIYMRMKHVRAWIPRPMRNQNYLFKANPNYAGNSIKPTGTSRTAITTL'}
  summarize_protein_role: {'protein_name': 'Genom

### What We Built

In this notebook, we defined tools for retrieving a protein sequence, calculating sequence-derived properties, summarizing a protein's role, and submitting an answer. We placed those tools in an Aviary environment, defined an LDP agent to choose among them, and used a rollout to record the resulting actions and observations.

The trajectory lets us check the agent's path through the task: which protein entity it selected from 9RIP, what sequence the retrieval tool returned, and what sequence the analysis tool received. This matters because a PDB entry can contain several proteins, and a plausible final answer does not show where its inputs came from.

The environment's reward marks answer submission, not scientific correctness. Sequence calculations describe the supplied amino acid sequnce; the role summary and proposed targeting ideas still beed to be checked against biological evidence. When building a research agent, inspect both its final answer and the tool results that support it.

### Further Reading

- [Aviary: Building a Custom Environment](https://github.com/Future-House/aviary/blob/main/tutorials/Building%20a%20Custom%20Environment%20in%20Aviary.ipynb) — another example of defining tools, observations, and environment steps.
- [LDP documentation and examples](https://github.com/Future-House/ldp) — agent interfaces and running an agent in an Aviary environment.
- [RCSB PDB Data API](https://data.rcsb.org/) — retrieving records for the specific protein entities in a PDB entry.
- [Biopython ProtParam](https://biopython.org/wiki/ProtParam) — the sequence calculations used in this notebook.

## ! Previously Broken Outputs and Query Shim !

In [17]:
# Print the tool calls from the last step of the last trajectory
tool_calls = trajectories[-1].steps[-1].action.value.tool_calls

# Get the answer from submit_final_answer tool call
for tc in tool_calls:
    if tc.function.name == "submit_final_answer":
        print(tc.function.arguments["answer"])

The protein with PDB ID 9RIP is currently not characterized, and there is no available data on its biological function or association with any disease. Therefore, it is not regarded as a drug target at this time. 

However, the analysis of its sequence reveals the following biophysical properties:
- Length: 34 amino acids
- Molecular Weight: 4044.48 Da
- Isoelectric Point: 5.51
- Instability Index: 48.74 (indicating it is likely unstable)
- GRAVY Score: -0.844 (suggesting it is hydrophilic)

Given its instability and hydrophilic nature, potential small-molecule drug targeting strategies could involve:
1. **Stabilization**: Developing small molecules that can stabilize the protein structure, potentially enhancing its function or stability.
2. **Modulation of Interactions**: Identifying and targeting any potential interaction partners that may be influenced by the protein, even if its direct function is unknown.
3. **Screening for Ligands**: Conducting high-throughput screening of small 

In [24]:
for i, step in enumerate(trajectories[0].steps, start=1):
    print(f"\nStep {i} — requested tools")
    for call in step.action.value.tool_calls:
        print(f"  {call.function.name}: {call.function.arguments}")

    print("Recorded observations:")
    for field in ("next_obs", "next_observation", "observation", "obs"):
        if hasattr(step, field):
            messages = getattr(step, field)
            print(f"  {field}:")
            for message in messages:
                print(f"    {getattr(message, 'name', type(message).__name__)}:")
                print(f"    {message.content}")


Step 1 — requested tools
  summarize_protein_role: {'protein_name': '9RIP', 'organism': 'unknown'}
Recorded observations:
  next_observation:
    summarize_protein_role:
    The protein 9RIP is not characterized, and there is no available data on its biological function. Consequently, its association with any disease or condition remains unknown. Without functional insights, it is not currently regarded as a drug target. Further research is necessary to determine its potential roles and relevance in medical contexts.
  observation:
    Message:
    
You are an expert researcher. You are given a research question, Your task is to answer the question. You have access to the following tools:
- analyze_protein_sequence: to analyze the protein
- summarize_protein_role: to summarize the biological role of the protein
- submit_final_answer: to submit the final answer

    Message:
    What is the biological role of the protein with PDB id 9RIP? Please analyze this protein and help me think a

In [22]:
sequence = (
    "MVGSLNCIVAVSQNMGIGKNGDLPWPPLRNEFRYFQRMTTTSSVEGKQNLVIMGKKTWFS"
    "IPEKNRPLKGRINLVLSRELKEPPQGAHFLSRSLDDALKLTEQPELANKVDMVWIVGGSS"
    "VYKEAMNHPGHLKLFVTRIMQDFESDTFFPEIDLEKYKLLPEYPGVLSDVQEEKGIKYKF"
    "EVYEKND"
)

new_query = f"""
The following amino acid sequence is human dihydrofolate reductase
(DHFR; UniProt P00374):

{sequence}

Calculate its sequence-derived properties, summarize DHFR's biological
role, and suggest questions we would need to investigate before
proposing a small-molecule targeting strategy.
"""

new_trajectories: list[tuple] = await runner.sample_trajectories(
    environments=[DemoAgentEnv(query=new_query)], # must be a list of environments. Can add multiple environments to run in parallel
    max_steps = 5, # max number of steps to run for each environment
)

new_tool_calls = new_trajectories[-1].steps[-1].action.value.tool_calls

# Get the answer from submit_final_answer tool call
for tc in new_tool_calls:
    if tc.function.name == "submit_final_answer":
        print(tc.function.arguments["answer"])

new_trajectory = new_trajectories[0]

for i, step in enumerate(new_trajectory.steps, start=1):
    action = step.action.value if step.action is not None else None
    calls = getattr(action, "tool_calls", None) or []
    print(f"Step {i}: {[call.function.name for call in calls]}")
    print(f"  done={step.done}, truncated={step.truncated}, reward={step.reward}")

The human dihydrofolate reductase (DHFR) has the following sequence-derived properties:
- Length: 187 amino acids
- Molecular Weight: 21,452.48 Da
- Isoelectric Point: 6.85
- Instability Index: 27.16 (indicating it is stable)
- GRAVY Score: -0.464 (indicating it is hydrophilic)
- Amino Acid Composition:
  - A: 2.7%
  - C: 0.5%
  - D: 4.8%
  - E: 8.6%
  - F: 4.8%
  - G: 7.0%
  - H: 1.6%
  - I: 4.8%
  - K: 9.1%
  - L: 10.2%
  - M: 3.7%
  - N: 5.3%
  - P: 6.4%
  - Q: 3.7%
  - R: 4.3%
  - S: 6.4%
  - T: 3.7%
  - V: 7.5%
  - W: 1.6%
  - Y: 3.2%

Biological Role of DHFR:
Dihydrofolate reductase (DHFR) in Homo sapiens is an enzyme involved in the folate pathway, catalyzing the reduction of dihydrofolate to tetrahydrofolate, which is essential for DNA synthesis and repair. It is associated with diseases such as cancer, where enhanced DHFR activity supports rapid cell proliferation, and infective conditions like bacterial infections, as some pathogens rely on DHFR. Due to its critical role in c